# Etapa 02 — diagnóstico e contrato da Silver

Autor: Wanderson Ferreira. Diagnóstico completo informado em 2026-10-06.
Silver `silver_v1` construída e verificada pelo autor sobre o histórico completo.

A síntese está no [notebook principal](../fraud_detection_mlops.ipynb). O
[contrato vigente](../../docs/SILVER_CONTRACT.md) define tipos e aceitação;
[Operations](../../docs/OPERATIONS.md#construir-e-verificar-a-silver) contém os
comandos. O contrato incorpora a decisão vigente; o Git preserva o rascunho
anterior e este notebook mantém o diagnóstico inicial.
As células de código abaixo são opcionais, sem outputs pré-fabricados.


## Inspeção inicial fornecida pelo autor

Em 2026-10-06, o autor conferiu a integridade de 183/183 arquivos e leu
`2018-04-01.pkl` com pandas 3.0.6. Foram informadas 9.488 linhas e nove colunas,
sem nulos, linhas duplicadas ou IDs de transação repetidos na partição.
Há 9.485 rótulos 0 e três rótulos 1, aproximadamente 0,0316% de fraude no dia.

O intervalo informado é 00:00:31 a 23:59:51. A média do valor é 53,249981,
o mínimo 0,13 e o máximo 226,40. Os resultados são do terminal do autor,
registrados no [recibo](../../references/evidence/silver_first_partition_2026-10-06.json).
Não são outputs pré-preenchidos das células deste notebook.


## O que a primeira partição indica

`CUSTOMER_ID`, `TERMINAL_ID`, `TX_TIME_SECONDS` e `TX_TIME_DAYS` guardam
inteiros Python com dtype `object`. A Silver deve ter um schema explícito,
com conversão verificada, em vez de depender de inferência de tipos.

A ausência de duplicatas no dia não comprova unicidade global. A frequência
de fraude nesse dia não representa todo o histórico. Valores extremos podem
conter informação importante para fraude e não serão eliminados automaticamente.

`TX_FRAUD_SCENARIO` registra a geração do rótulo na simulação: será metadado de
auditoria e ficará fora das entradas do futuro modelo. O contrato deve explicitar
a convenção temporal, pois timestamps observados não têm timezone.


In [1]:
import json
from pathlib import Path

candidates = (Path.cwd(), *Path.cwd().parents)
project_root = next(
    (path for path in candidates if (path / "references/handbook_source.json").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Execute a partir do checkout do projeto.")
inventory = json.loads(
    (project_root / "references/handbook_source.json").read_text(encoding="utf-8")
)
evidence = json.loads(
    (project_root / "references/evidence/silver_first_partition_2026-10-06.json").read_text(
        encoding="utf-8"
    )
)
print("Escopo:", evidence["scope"])
print("Partição:", evidence["partition"])
print("Tipos informados:", evidence["raw_dtypes"])

Escopo: first_partition_diagnostic; not_full_dataset_semantic_validation
Partição: 2018-04-01.pkl
Tipos informados: {'TRANSACTION_ID': 'int64', 'TX_DATETIME': 'datetime64[ns]', 'CUSTOMER_ID': 'object', 'TERMINAL_ID': 'object', 'TX_AMOUNT': 'float64', 'TX_TIME_SECONDS': 'object', 'TX_TIME_DAYS': 'object', 'TX_FRAUD': 'int64', 'TX_FRAUD_SCENARIO': 'int64'}


## Auditoria dos 183 arquivos

O autor executou a CLI do profiler e informou **1.754.155 linhas e IDs distintos**,
14.681 fraudes e 1.739.474 transações genuínas. Não foram encontradas violações
nos controles implementados; houve 42 valores monetários zero. A frequência
global de fraude é aproximadamente 0,8369%, diferente da frequência do primeiro dia.

O [recibo completo](../../references/evidence/silver_profile_2026-10-06.json) é uma
transcrição do resumo do terminal; o JSON nativo não foi fornecido para inspeção.
Os contadores podem se sobrepor, não são uma contagem única de linhas inválidas.
A próxima célula lê o relatório local existente; não executa aquisição ou transformação.


In [2]:
import hashlib

profile_path = (
    project_root / "data/interim/handbook" / inventory["source"]["commit"] / "profile.json"
)
if not profile_path.is_file():
    raise FileNotFoundError(
        "Execute primeiro: poetry run python -m fraud_detection_mlops.profiling"
    )
report = json.loads(profile_path.read_text(encoding="utf-8"))
if report["source"]["commit"] != inventory["source"]["commit"]:
    raise ValueError("O perfil pertence a outra fonte.")
current_inventory_hash = hashlib.sha256(
    (project_root / "references/handbook_source.json").read_bytes()
).hexdigest()
if report["inventory_sha256"] != current_inventory_hash:
    raise ValueError("O inventário difere daquele usado no perfil.")
print("Geração UTC:", report["generated_at_utc"])
print("Ambiente:", report["environment"])
print(json.dumps(report["summary"], ensure_ascii=False, indent=2))

Geração UTC: 2026-10-07T02:10:46.436018+00:00
Ambiente: {'python': '3.14.4', 'pandas': '3.0.6'}
{
  "verified_files": 183,
  "profiled_files": 183,
  "empty_partitions": 0,
  "rows": 1754155,
  "distinct_valid_transaction_ids": 1754155,
  "fraud_count": 14681,
  "genuine_count": 1739474,
  "checks": {
    "null_values": 0,
    "duplicate_rows": 0,
    "duplicate_tx_ids_within_partition": 0,
    "duplicate_tx_ids_across_previous_partitions": 0,
    "negative_identifier_values": 0,
    "invalid_timestamps": 0,
    "timestamps_outside_partition": 0,
    "timestamp_order_decreases": 0,
    "seconds_datetime_mismatch": 0,
    "days_datetime_mismatch": 0,
    "invalid_fraud_labels": 0,
    "invalid_scenario_labels": 0,
    "fraud_scenario_mismatch": 0,
    "invalid_or_nonfinite_amounts": 0,
    "negative_amounts": 0,
    "zero_amounts": 42
  },
  "invalid_int64_by_column": {
    "TRANSACTION_ID": 0,
    "CUSTOMER_ID": 0,
    "TERMINAL_ID": 0,
    "TX_TIME_SECONDS": 0,
    "TX_TIME_DAYS": 0,


## Decisões após o diagnóstico completo

O [gerador do Handbook](https://fraud-detection-handbook.github.io/fraud-detection-handbook/Chapter_3_GettingStarted/SimulatedDataset.html)
arredonda os valores monetários para duas casas decimais. Zero é compatível com
esse processo; a causa individual dos 42 casos não foi rastreada. A Silver aceita
valores finitos maiores ou iguais a zero e preserva esses registros e os extremos.

Quatro colunas de inteiros com dtype `object` passam para `int64`; os dois rótulos
passam para `int8`. Timestamp mantém nanossegundos e ausência de timezone. Não há
imputação, deduplicação, filtro de linhas ou criação de features. Cenário de fraude
é metadado de auditoria e não será entrada preditiva.

O builder recalcula o diagnóstico, escreve os Parquets em staging, confere a
leitura de volta e reconcilia as contagens com SQL antes de publicar o diretório
completo. Manifesto e auditoria associam entrada, saída, contrato e código por hash.
Repetições verificam e reutilizam o dataset íntegro. O
[runbook](../../docs/OPERATIONS.md#construir-e-verificar-a-silver) explica a operação.

A execução completa relatada pelo autor está registrada abaixo. A célula de código
apenas verifica e consulta a Silver já construída localmente.
A EDA que orientar features e ajustes do modelo respeitará as janelas temporais
para preservar a avaliação final. Nesta etapa não há avaliação de modelo.


## Validação da implementação preparada

Em ambiente de preparação, Python 3.14.4 e pandas 3.0.6, passaram 70 testes.
A primeira partição real também foi convertida e verificada: 9.488 linhas e IDs
únicos, três fraudes e 9.485 genuínas. O inventário desse ensaio foi limitado a um
dia; isso não valida a conversão do histórico completo. As evidências e seus limites
estão no [recibo da implementação](../../references/evidence/silver_implementation_validation_2026-10-07.json).

O ensaio real revelou um detalhe de compatibilidade: os pickles antigos preservam
o índice dos nomes das colunas com dtype `object`; pandas 3 e Arrow retornam nomes
com o tipo atual de string. A normalização recria esse índice com os mesmos nove
nomes, sem mudar os dados. Os testes controlados incluem essa representação antiga,
assim como índices de linhas que não começam em zero.

Os testes também cobrem zeros preservados, duplicatas entre dias, entradas inválidas,
falha de escrita, alteração da Bronze entre verificação e leitura, corrupção de
saída, divergência de schema/contagens, reutilização e lock concorrente.
A execução completa posteriormente relatada pelo autor está registrada a seguir.


## Silver completa — resultado informado pelo autor

Em 2026-10-06, após aplicar `silver-parquet.patch`, o autor executou `build` com
`status: success` e depois `verify`, ambos com as mesmas contagens:

| Medida | Resultado observado no terminal |
| --- | --- |
| Partições verificadas | 183 |
| Linhas | 1.754.155 |
| IDs distintos | 1.754.155 |
| Fraudes | 14.681 |
| Genuínas | 1.739.474 |
| Valores zero | 42 |

Todas as contagens reconciliam com o diagnóstico completo da Bronze, com nenhuma
linha removida. A auditoria responsável é
`silver_f9fc5a5083f14d9fadf6892a531e1488.json`. Ambiente/lockfile, lint, formatação
e diff passaram; o pytest informou 70 testes aprovados em 0,91 s.

O [recibo da execução](../../references/evidence/silver_build_2026-10-06.json)
transcreve essas saídas e normaliza os caminhos relativos ao projeto. Os JSONs
nativos e os Parquets permanecem locais; não foram inspecionados aqui. Nenhuma
saída foi inserida como execução de uma célula deste notebook. A implementação foi publicada em `acf1516`, com
[CI aprovada](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37563463479).

A Silver está validada no escopo dos controles implementados. A próxima etapa
é EDA e desenho da separação temporal, seguida de features na Gold e baseline.
Estatísticas que orientem o modelo devem respeitar o período de treinamento.


In [3]:
from fraud_detection_mlops.silver import connect_silver, verify_silver

silver_root = project_root / "data/interim/handbook"
inventory_path = project_root / "references/handbook_source.json"
contract_path = project_root / "references/silver_contract_v1.json"
print(verify_silver(silver_root, inventory_path=inventory_path, contract_path=contract_path))
with connect_silver(
    silver_root, inventory_path=inventory_path, contract_path=contract_path
) as connection:
    display(
        connection.execute("""
        SELECT tx_date, count(*) AS transactions, sum(TX_FRAUD) AS frauds
        FROM transactions
        GROUP BY tx_date
        ORDER BY tx_date
        LIMIT 7
    """).fetchdf()
    )

2026-10-07 10:44:08.654 | INFO     | fraud_detection_mlops.config:<module>:11 - PROJ_ROOT path is: /home/wanderson/Documentos/fraud-detection-mlops


{'silver_path': '/home/wanderson/Documentos/fraud-detection-mlops/data/interim/handbook/6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a/silver_v1', 'verified_partitions': 183, 'rows': 1754155, 'distinct_transaction_ids': 1754155, 'fraud_count': 14681, 'genuine_count': 1739474, 'zero_amounts': 42}


,tx_date,transactions,frauds
0,2018-04-01,9488,3.0
1,2018-04-02,9583,13.0
2,2018-04-03,9747,15.0
3,2018-04-04,9530,18.0
4,2018-04-05,9651,22.0
5,2018-04-06,9539,27.0
6,2018-04-07,9438,39.0
